In [ ]:
%load_ext sagemaker_studio_analytics_extension.magics
%sm_analytics emr connect --verify-certificate False --cluster-id j-28EBBTTI1J3HN --auth-type None --language python  

INFO:root:Read [/opt/ml/metadata/resource-metadata.json] from disk at 1777415427287840567
INFO:root:Returning content of [/opt/ml/metadata/resource-metadata.json] updated at time 1777415427287840567
INFO:root:Read [/opt/.sagemakerinternal/internal-metadata.json] from disk at 1777415411257843459


Successfully read emr cluster(j-28EBBTTI1J3HN) details


INFO:sagemaker_studio_sparkmagic_lib.emr:Successfully read emr cluster(j-28EBBTTI1J3HN) details


Initiating EMR connection..
Starting Spark application


##タスク 6: SparkMagic PySpark カーネルでデータを探索し、クエリを実行する

このノートブックでは、Amazon EMR クラスターに接続した状態で SparkMagic PySpark を使用してデータを探索してクエリを実行します。

### タスク 6.1: セッション情報を表示する

PySpark カーネルを使用するため、PySpark magic の %%info を使用して現在のセッション情報を表示できます。

In [3]:
%%info

An error was encountered:
Error sending http request and maximum retry encountered.


### タスク 6.2: データを探索し、そのデータに対してクエリを実行する

 PySpark カーネルにはすでに一般的な SQLContext が追加されていますが、ここでは HiveContext と呼ばれる SQLContext のサブセットを使用していることに注意してください。SQLContext は構造化データを扱うためのより汎用的なインターフェイスですが、HiveContext は Hive およびそのエコシステムに固有のものです。SQLContext はより広いデータソースをサポートしていますが、HiveContext は Hive テーブルとメタストアに特化しており、Hive UDF、Hive インデックス作成、Hive 統計のサポートなどの追加機能を提供します。

 PySpark カーネルを使用すると、EMR クラスターへの接続後に SparkContext および HiveContext が自動的に作成されます。HiveContext を使用して Hive テーブルのデータに対してクエリを実行し、そのデータを Spark データフレームで使用できます。

HiveContext を使用して Hive に対してクエリを実行し、データベースとテーブルを確認します。

In [2]:
#query-hive

sqlContext = HiveContext(sqlContext)

dbs = sqlContext.sql("show databases")
dbs.show()

tables = sqlContext.sql("show tables")
tables.show()

The code failed because of a fatal error:
	Error sending http request and maximum retry encountered..

Some things to try:
a) Make sure Spark has enough available resources for Jupyter to create a Spark context.
b) Contact your Jupyter administrator to make sure the Spark magics library is configured correctly.
c) Restart the kernel.


Adult データテーブルに対してクエリを実行し、データを Spark データフレームに取り込みます。

In [ ]:
#load-data

adult_df = sqlContext.sql("select * from adult_data").cache()

データフレームを使用して形状を確認し、データセットの最初の 5 行を表示します。

In [ ]:
#view-shape

print((adult_df.count(), len(adult_df.columns)))

#Show first 5 rows 
adult_df.head(5)

出力を見やすくするために、Spark データフレームを Pandas データフレームに変換します。

In [ ]:
#convert-dataframe

adult_df.limit(5).toPandas()

線形回帰などの一部の機械学習 (ML) アルゴリズムでは、数値型の特徴が必要です。このラボで使用する Adult データセットには、**workclass**、**education**、**occupation**、**marital status**、**relationship**、**race**、**sex** などのカテゴリ型の特徴量が含まれています。

次のコードブロックで、StringIndexer および OneHotEncoderEstimator を使用して、カテゴリ型の変数を 0 と 1 の値を取る数値型の変数のセットに変換する方法を示します。

- StringIndexer は文字列値の列をラベルインデックスの列に変換します。
- OneHotEncoderEstimator はカテゴリインデックスの列をバイナリベクトルの列にマッピングします。行のカテゴリインデックスを示す「1」が各行に最大 1 つあります。

Spark のワンホットエンコーディングは 2 ステップのプロセスです。まず **StringIndexer** を使用し、その後 **OneHotEncoder** を使用します。

詳細については、「[StringIndexer](https://spark.apache.org/docs/latest/ml-features.html#stringindexer)」および「[OneHotEncoder](https://spark.apache.org/docs/latest/ml-features.html#onehotencoder)」を参照してください。

In [ ]:
#convert-variables

from pyspark.ml.feature import StringIndexer, OneHotEncoderEstimator

categorical_variables = ['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country']

indexers = [StringIndexer(inputCol=column, outputCol=column+"-index") for column in categorical_variables]

encoder = OneHotEncoderEstimator(
    inputCols=[indexer.getOutputCol() for indexer in indexers],
    outputCols=["{0}-encoded".format(indexer.getOutputCol()) for indexer in indexers]
)

VectorAssembler クラスは、複数の列を入力として受け取り、値の配列を含む 1 つの列を出力します。

このアセンブラの詳細については、「[VectorAssembler](https://spark.apache.org/docs/latest/ml-features.html#vectorassembler)」を参照してください。

In [ ]:
#vector-assembler

from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=encoder.getOutputCols(),
    outputCol="categorical-features"
)

パイプラインは、変換器と推定器の順序ありリストです。データセットに適用される変換処理を自動化し、再現性を確保するためにパイプラインを定義します。このステップでは、パイプラインを定義してデータセットに適用します。

**StringIndexer** と同様に、パイプラインは**推定器**です。pipeline.fit() メソッドは**変換器**である **PipelineModel** を返します。

機械学習パイプラインの詳細については、「[Pipelines](https://spark.apache.org/docs/latest/ml-pipeline.html#pipeline)」を参照してください。

In [ ]:
#pyspark-pipelines

from pyspark.ml import Pipeline

# Define the pipeline based on the stages created in previous steps.
pipeline = Pipeline(stages=indexers + [encoder, assembler])

# Define the pipeline model.
pipelineModel = pipeline.fit(adult_df)

# Apply the pipeline model to the dataset.
adult_df = pipelineModel.transform(adult_df)

前のステップで作成したさまざまな列をすべて確認します。

In [ ]:
#print-schema

adult_df.printSchema()

変換の適用後は、エンコードされたカテゴリ型の変数をすべて含む配列が 1 つの列に格納されます。

In [ ]:
#view-categorical-features

adult_df.select('categorical-features').show(truncate=False)

ここで、ターゲットラベルをエンコードします。

In [ ]:
#encode-target

indexer = StringIndexer(inputCol='income', outputCol='label')

adult_df = indexer.fit(adult_df).transform(adult_df)

### タスク 6.3: Spark UI を使用してモニタリングとデバッグを行う

このセクションでは、Spark UIを使用して、前のステップで実行した Spark ジョブのパフォーマンスをモニタリングおよび検査します。

現在の Spark セッション情報を取得します。

In [ ]:
%%info

**Spark UI** と **Driver log** のハイパーリンクが見つかります。このラボでは、**Driver log** のリンクは無効になっています。**Spark UI** の署名付き URL は、EMR クラスターへの接続時に生成されます。このリンクをクリックすると、Spark UI が表示され、ウェブブラウザで Spark ジョブの実行結果について調べることができます。Spark UI に表示されるメトリクスは、パフォーマンスのチューニングに役立ちます。

Spark サーバーで注意すべき重要な機能を次にいくつか示します。
- [**Jobs**] タブに、Spark アプリケーションのすべての Spark ジョブのステータスが表示されます。
- [Summary] セクションの [**Event Timeline**] セクションに、実行のさまざまなステージが表示されます。
- [**Completed Jobs**] セクションは表形式で表示されます。**[Completed Jobs**] セクションでは、ジョブを選択して、その中のタスクのステージに関する情報を確認できます。
- [**DAG Visualization**] を使用すると、過去に実行されたタスクを調べることができます。[**Event Timeline**] ビューと同様、[**DAG visualization**] を使用すると、ステージを選択してそのステージ内の詳細を表示できます。

### クリーンアップ

このノートブックを完了しました。ラボの次の部分に進むために、以下を実行してください。

- このノートブックファイルを閉じる。
- ラボのセッションに戻り、「**まとめ**」を続ける。